# DeepGaze scanpath inference in Google Colab

This notebook runs the two trained tasks used by the scanpath-estimation app:

1. Free viewing with the `combined_adapter`
2. Searching for a car with the `visual_search_adapter`

The prompt templates are imported from this project so they remain byte-identical to the validated versions. The outputs are written as JSON files compatible with the web app's `data/model/` folder.

In [ ]:
# Colab setup: use a GPU runtime before running this cell.
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y git-lfs
!git lfs install
!pip -q install -r https://raw.githubusercontent.com/berbelekhieronim/scanpath-estimation/claude/eye-gaze-annotation-app-fvoz8p/requirements-model.txt

In [ ]:
# Fetch the exact prompt/parser code and the trained DeepGaze adapters.
%cd /content
!rm -rf scanpath-estimation DeepGaze3.5-VL
!git clone --depth 1 --branch claude/eye-gaze-annotation-app-fvoz8p https://github.com/berbelekhieronim/scanpath-estimation.git
!git clone https://github.com/Susmit-A/DeepGaze3.5-VL.git
!cd DeepGaze3.5-VL && git lfs pull

import sys
sys.path.insert(0, '/content/scanpath-estimation/tools')
import gaze_prompts as gp

print('Prompt/parser code loaded.')
print('Base model:', gp.BASE_MODEL)

## Choose an image

Upload one JPG, JPEG, PNG, or WEBP image. The model coordinates use a 0-99 grid where `(0, 0)` is the top-left.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload an image before continuing.')

image_name = next(iter(uploaded))
image_path = Path('/content') / image_name
print(f'Using: {image_path}')

## Configure the runs

Five fixations match the participant task in the demo. Increase `samples` for multiple virtual observers; `temperature=0.7` produces varied sampled paths. Set `samples=1` and `temperature=0.0` for one deterministic path.

In [ ]:
NUM_FIXATIONS = 5
SAMPLES = 10
TEMPERATURE = 0.7
SEED = 42
DEVICE = 'cuda'
DTYPE = 'float16'

repo_path = '/content/DeepGaze3.5-VL'
output_dir = Path('/content/scanpath_outputs')
output_dir.mkdir(exist_ok=True)

freeview_prompt = gp.build_freeview_prompt(NUM_FIXATIONS)
car_prompt = gp.build_search_prompt('car', NUM_FIXATIONS)
print('FREEVIEW PROMPT')
print(freeview_prompt)
print('\nCAR SEARCH PROMPT')
print(car_prompt)

In [ ]:
# Load the project inference implementation. It uses plain Transformers + PEFT.
sys.path.insert(0, '/content/scanpath-estimation/tools')
import predict_mps
from PIL import Image
import gc
import json
import time

image = Image.open(image_path).convert('RGB')

def run_task(mode, target, adapter, prompt_text, output_name):
    print(f'Loading {adapter} for {mode}...')
    model, processor = predict_mps.load_model(
        repo_path, adapter, DEVICE, DTYPE
    )
    started = time.time()
    texts = predict_mps.predict(
        model, processor, image, prompt_text, SAMPLES, TEMPERATURE,
        SEED, DEVICE, max(64, 16 * NUM_FIXATIONS + 16)
    )
    samples_grid = [
        parsed for parsed in (gp.parse_scanpath(text) for text in texts)
        if parsed
    ]
    if not samples_grid:
        raise RuntimeError(f'No parseable coordinates returned: {texts}')

    elapsed = time.time() - started
    result = {
        'image': image_path.name,
        'mode': mode,
        'target': target,
        'n_fixations': NUM_FIXATIONS,
        'seed': SEED,
        'temperature': TEMPERATURE,
        'prompt_text': prompt_text,
        'prompt_kind': 'trained',
        'scanpath_grid': samples_grid[0],
        'scanpath_norm': gp.grid_to_norm(samples_grid[0]),
        'samples_grid': samples_grid,
        'samples_norm': [gp.grid_to_norm(path) for path in samples_grid],
        'source': 'precomputed',
        'model': f'{gp.BASE_MODEL} + {adapter}',
        'device': DEVICE,
        'elapsed_seconds': round(elapsed, 1),
        'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    output_path = output_dir / output_name
    output_path.write_text(json.dumps(result, indent=2))
    print(f'Wrote {len(samples_grid)} path(s) in {elapsed / 60:.1f} min: {output_path}')
    print('First path:', samples_grid[0])

    # Only one 8B model copy needs to stay in GPU memory at a time.
    del model, processor
    gc.collect()
    import torch
    torch.cuda.empty_cache()
    return result

## Run 1: free viewing

This loads the `combined_adapter` and uses the exact trained free-viewing prompt.

In [ ]:
freeview_result = run_task(
    mode='freeview',
    target=None,
    adapter='combined_adapter',
    prompt_text=freeview_prompt,
    output_name=f'{image_path.stem}__freeview__n{NUM_FIXATIONS}.json',
)

## Run 2: search for a car

This reloads the `visual_search_adapter` and uses the exact trained COCO-Search18 `car` prompt.

In [ ]:
car_result = run_task(
    mode='search',
    target='car',
    adapter='visual_search_adapter',
    prompt_text=car_prompt,
    output_name=f'{image_path.stem}__search__car__n{NUM_FIXATIONS}.json',
)

In [ ]:
# Download both JSON files, or copy them into the app's data/model/ folder.
from google.colab import files
import shutil

archive_path = shutil.make_archive('/content/scanpath_outputs', 'zip', output_dir)
print(f'Created: {archive_path}')
files.download(archive_path)

### Using the results in the web app

Unzip the downloaded archive and copy both JSON files into the app's `data/model/` directory. Then use **Reload runs from disk** in `/control`, or restart the app.

The model is an 8B vision-language model and needs substantial GPU memory. A Colab GPU may run out of memory; if that happens, use a higher-memory runtime, reduce `SAMPLES`, or set `DTYPE = 'bfloat16'` on a GPU that supports it.